## Out of vocabulary setup 
Measures how badly the base tokenizer fragments frequent fan vocabulary. Scores words by frequency and sub-word splits to build the expansion shortlist for Notebook 04.

Input: cleaned corpus. Output: ranked OOV candidate tokens.


## 1. Setting file paths

In [ ]:
import os, re
from glob import glob
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer

os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")
os.makedirs(OUT := "data/processed", exist_ok=True)

# Full DAPT corpus
FILES = sorted(glob("data/interim/clean/*.parquet") + glob("data/interim/clean_dapt/*.parquet"))
print(f"{len(FILES)} files")

SAMPLE, MIN_FREQ = 300_000, 20
# texts to sample for frequency counting (speed vs coverage), ignores words appearing fewer than this many times

38 files


## Building vocabulary frequency table

In [4]:
# Sample texts across all subreddits proportionally, count word frequencies.
WORD = re.compile(r"[a-z][a-z'\-]+")   # lowercase alpha words, keeps don't / bias-wrecker

counts = Counter()
per_file = max(SAMPLE // len(FILES), 500)

for p in tqdm(FILES, desc="counting"):
    d = pd.read_parquet(p, columns=["text"])
    txt = d.text.dropna()
    if len(txt) > per_file:
        txt = txt.sample(per_file, random_state=0)
    for t in txt:
        counts.update(WORD.findall(t.lower()))
    del d

print("distinct words:", len(counts))
freq = pd.DataFrame(counts.items(), columns=["word", "freq"])
freq = freq[freq.freq >= MIN_FREQ].sort_values("freq", ascending=False)
print("words above min_freq:", len(freq))

counting: 100%|██████████| 38/38 [00:04<00:00,  7.93it/s]

distinct words: 123180
words above min_freq: 15955


## 03_oov status

**Corpus:** 38 files, 15.9k words (min_freq=20)
**Yield:** Clean splits (≤2): base __% | twt __%. 85% OOV mass = __ tokens.

**Curation:** Keep slang/ritual/harm. Drop names/typos/users.

* [ ] Curate `oov_candidates.csv`
* [ ] Run cell 6 → `new_tokens.csv` (__ tokens, feeds 04_dapt)

In [ ]:
tok_base = AutoTokenizer.from_pretrained("xlm-roberta-base")
tok_tw = AutoTokenizer.from_pretrained("cardiffnlp/twitter-xlm-roberta-base")

# Prefix to simulate mid-sentence word starts. 
spaced = [" " + w for w in freq.word]

freq["frag_base"] = [len(ids) for ids in tok_base(spaced, add_special_tokens=False).input_ids]
freq["frag_tw"] = [len(ids) for ids in tok_tw(spaced, add_special_tokens=False).input_ids]

# Mass = frequency * fragments (tokenizer waste). High mass = add to vocab.
freq["mass_base"] = freq.freq * freq.frag_base
freq["mass_tw"] = freq.freq * freq.frag_tw

freq.to_parquet(f"{OUT}/word_freq_frag.parquet")

twitter: 100%|██████████| 15955/15955 [00:00<00:00, 44401.85it/s]


## 3. Tokenizer coverage

Compares `xlm-roberta-base` against a Twitter-adapted variant to see which fandom terms each already handles cleanly.

In [6]:
# How much does the social-media model already cover vs the base?
covered_base = (freq.frag_base <= 2).mean()
covered_tw   = (freq.frag_tw <= 2).mean()
print(f"words tokenised cleanly (<=2 pieces):")
print(f"  xlm-roberta-base      : {covered_base:.1%}")
print(f"  twitter-xlm-roberta   : {covered_tw:.1%}")

# words the Twitter model handles but the base doesn't -> social pretraining helps
tw_helps = freq[(freq.frag_base >= 3) & (freq.frag_tw <= 2)]
print(f"\nterms Twitter model covers that base fragments: {len(tw_helps)}")
print(tw_helps.head(20)[["word","freq","frag_base","frag_tw"]].to_string(index=False))

words tokenised cleanly (<=2 pieces):
  xlm-roberta-base      : 80.0%
  twitter-xlm-roberta   : 80.0%

terms Twitter model covers that base fragments: 0
Empty DataFrame
Columns: [word, freq, frag_base, frag_tw]
Index: []


In [ ]:
# Rank OOV by waste removed. Cumulative mass flags the 85% proposal target.
oov = freq[freq.frag_base >= 3].sort_values("mass_base", ascending=False).copy()
oov["cum_frac"] = oov.mass_base.cumsum() / oov.mass_base.sum()

cols = ["word", "freq", "frag_base", "frag_tw", "mass_base", "cum_frac"]
cut85 = oov[oov.cum_frac <= 0.85][cols].assign(keep="", domain="", note="")

print(f"{len(cut85)} tokens cover 85% OOV mass. Total OOV: {len(oov)}\n")
cut85.to_csv(f"{OUT}/oov_candidates.csv", index=False)
print("Wrote oov_candidates.csv for manual curation:")
print(cut85.head(30).to_string(index=False))

829 tokens cover 85% of OOV mass (target from proposal)
full OOV-ish list: 3195 tokens

wrote oov_candidates.csv — curate by hand next
      word  freq  frag_base  frag_tw  mass_base  cum_frac keep domain note
      it's 52486          3        3     157458  0.079529                 
       i'm 40924          3        3     122772  0.141538                 
     don't 40611          3        3     121833  0.203073                 
    that's 14485          3        3      43455  0.225021                 
      i've 14481          3        3      43443  0.246963                 
   they're 13413          3        3      40239  0.267287                 
     can't 12950          3        3      38850  0.286909                 
    didn't 11630          3        3      34890  0.304531                 
   doesn't 11134          3        3      33402  0.321402                 
    you're  8372          3        3      25116  0.334088                 
     she's  8279          3        3    

## 4. Finalising the candidate token list

The out-of-vocabulary candidates are ranked by how much tokenizer waste their addition would remove, and the shortlist is saved for the vocabulary expansion in Notebook 04.

In [ ]:
import pandas as pd, os

os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")
PROC = "data/processed"

d = pd.read_csv(f"{PROC}/oov_candidates.csv")
d["keep"] = pd.to_numeric(d["keep"], errors="coerce")
wl = d.word.str.lower()

# Bulk overrides
d.loc[wl == "k-pop", "domain"] = "general"
d.loc[wl == "engenes", "domain"] = "parasocial"
d.loc[wl.isin(["ni-ki", "lmfao", "prettiest"]), "keep"] = 0

# Standardise domains
d["domain"] = d["domain"].astype(str).str.strip().replace("financial/parasocial", "financial")
d.loc[d.domain == "generic", "keep"] = 0

keep = d[d.keep == 1]
print(f"Final keep: {len(keep)}\n{keep.domain.value_counts().to_string()}")

blanks = keep[keep.domain.isin(["nan", ""]) | keep.domain.isna()]
if len(blanks): print(f"WARNING - Missing domain: {len(blanks)}", blanks.word.tolist())

d.to_csv(f"{PROC}/oov_candidates_clean.csv", index=False)

tokens = pd.Series(keep.word.str.lower().unique(), name="token").sort_values()
tokens.to_csv(f"{PROC}/new_tokens.csv", index=False)
print(f"Wrote new_tokens.csv: {len(tokens)}")

final keep: 314
domain
general       147
victim         97
parasocial     43
financial      27
keeps missing domain: 0 []
wrote new_tokens.csv: 314
